In [1]:
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 8.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 9.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 7.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 7.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 14.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 9.9 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.2.3
    Uninstalling urllib3-2.2.3:
      Successfully uninstalled urllib3-2.2.3
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0
    Uninstalling python-dateutil-2.9.0:
      Successfully uninstalled python-dateutil-2.9.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled num

In [15]:
pip install tabulate

Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
import json
import boto3
import glob
from urllib.parse import unquote_plus
from pyspark.sql import SparkSession
import uuid
import sys
import psycopg2
import logging
import pandas as pd
import janitor
from datetime import datetime
from pandas.api.types import infer_dtype
from pyspark.sql.functions import col, trim, regexp_replace, row_number, sha2, concat_ws, conv, lit, concat, coalesce, isnan,  isnull, regexp_replace,  monotonically_increasing_id, format_string
import tempfile
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, DoubleType, StringType, TimestampType, LongType
import numpy as np
import hashlib
from pyspark.sql.functions import udf
import warnings
from typing import Dict, Optional
from tabulate import tabulate

In [ ]:
import csv
with open(inputh, 'r') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        if i == 9034:  # 0-based index equivalent to line 9035
            print(f"Columns: {len(row)}, Data: {row}")

In [2]:
inputtableA = 'tableA.csv'
inputtableB = 'tableB.csv'

In [3]:
tableA = pd.read_csv(inputtableA, header=0, sep=',',low_memory=False)
tableB = pd.read_csv(inputtableB, header=0, sep=',')

In [4]:
count_a = tableA.groupby('e3did')['record'].count().reset_index(name='count_a')
count_b = tableB.groupby('e3did')['record'].count().reset_index(name='count_b')

In [5]:
max_counts = pd.merge(count_a, count_b, on='e3did', how='outer').fillna(0)
max_counts['max_count'] = max_counts[['count_a', 'count_b']].max(axis=1)

In [7]:
expanded = []
for _, row in max_counts.iterrows():
    expanded.extend([(row['e3did'], i+1) for i in range(int(row['max_count']))])

In [8]:
expanded_df = pd.DataFrame(expanded, columns=['e3did', 'record'])

In [9]:
result= (
    expanded_df
    .merge(tableA, on=['e3did', 'record'], how='left')
    .merge(tableB, on=['e3did', 'record'], how='left')
)

In [10]:
final_df = result.sort_values(['e3did', 'record']).reset_index(drop=True)

In [19]:
print(final_df.columns)

Index(['e3did', 'record', 'total', 'date_execut', 'supportid', 'montaje',
       'fecha2'],
      dtype='object')


In [17]:
print(tabulate(final_df, headers='keys', tablefmt='psql'))

+----+-------------------------------------+----------+---------+-----------------+-----------------------------+-----------+----------------+
|    | e3did                               |   record |   total | date_execut     | supportid                   |   montaje | fecha2         |
|----+-------------------------------------+----------+---------+-----------------+-----------------------------+-----------+----------------|
|  0 | /A10001-100-AGL-10395-U1BC2A-CF_01  |        1 |    1    | 8/19/2024 0:00  | /800228-A10001-MP-STD-00559 |         1 | 3/19/2025 0:00 |
|  1 | /A10001-100-AGL-10395-U1BC2A-CF_01  |        2 |    1    | 8/19/2024 0:00  | /800228-A10001-MP-STD-00559 |         1 | 3/19/2025 0:00 |
|  2 | /A10001-100-AGL-10395-U1BC2A-CF_01  |        3 |    1    | 8/19/2024 0:00  | /800228-A10001-MP-STD-01306 |       nan | nan            |
|  3 | /A10001-100-AGL-10395-U1BC2A-CF_01  |        4 |    0.75 | 8/20/2024 0:00  | /800228-A10001-MP-STD-00571 |       nan | nan            |